### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="cardiotocography",
    dataset_year="2010",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="10.24432/C51S4N",
    download_description="""
wget https://archive.ics.uci.edu/static/public/193/cardiotocography.zip \
&& unzip cardiotocography.zip && rm cardiotocography.zip \
&& mkdir -p local-data-warehouse/cardiotocography \
&& mv CTG.xls local-data-warehouse/cardiotocography
""",
    # References
    academic_reference_bibtex="""@misc{campos2010cardiotocography,
  author       = {Campos, D. and Bernardes, J.},
  title        = {{Cardiotocography}},
  year         = {2000},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C51S4N}
}
""",
    academic_reference_bibtex_key="campos2010cardiotocography",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We use the data from UCI and the 3 class problem of predicting NSP as it is more medical relevant.

- We transform the file names into patient IDs to indicate the sub-group of samples recorded from the same patient.
- The raw data has dates and the start and end index of the recording. We drop these as they are not relevant for the predictive task and would not be available for test samples.
- We drop all other columns that represent the label.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="NSP",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    # For classification
    stratify_on="NSP",
    # For grouped data
    group_on="patient_id",
    group_labels="per_sample",
)

## Preprocessing

In [ ]:
import pandas as pd
import uuid


# Read excel file
df = pd.read_excel(dataset_mold.path / "CTG.xls", sheet_name="Raw Data")
print("Loaded data shape:", df.shape)

# -- Remove rows that are not data rows from Excel sheet
# Drop first row, which is missing in the Excel sheet
df = df.drop(index=0).reset_index(drop=True)
# Drop last three rows as they are also corrupted
df = df.drop(index=df.index[-3:]).reset_index(drop=True)


df['FileName'] = (
    df['FileName']
    .str.replace(r'\.[^.]+$', '', regex=True)
    .pipe(lambda s: s.where(s.str.fullmatch(r'S\d+'),
                           s.str.replace(r'(_?\d+)$', '', regex=True)))
)
# Create mapping: molecule -> random string id
mapping = {val: uuid.uuid4().hex[:12] for val in df["FileName"].unique()}
df["patient_id"] = df["FileName"].map(mapping)

df = df.drop(columns=[
    "FileName",
    # Duplicat of LB
    "LBE",
    # Non-feature information
    "b",
    "e",
    "Date",
    "SegFile",
    # Classes
    "A", "B", "C", "D", "E",
    "AD","DE","LD","FS","SUSP","CLASS",


])

as_cat_dtype = ["patient_id", "NSP"]
df[as_cat_dtype] = df[as_cat_dtype].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For Grouped Non-IID data
split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()